# S2 — in-silico visual localizer (TRIBE v2)Runs the **frozen** Phase C design. This notebook makes no experimental choices: everyparameter comes from `neurocheck/s2_design.py`.**Order matters. Do not skip ahead.**1. setup — clone the repo, install deps2. verify inputs — hashes must match the manifest3. **frame-sampling check** — decides whether the 8 fps stimulus is usable *at all*4. go/no-go5. run S26. compliance checkIf step 3 says **INDEX-based**, stop. Do not run step 5. The stimulus must bere-rendered at 16 fps on a machine you control, then re-uploaded.

## 1 · SetupSet `BRANCH` to the branch holding the S2 design.

In [ ]:
BRANCH = "main"                     # Phase A+B+C all landed on mainDATASET = None                      # e.g. "/kaggle/input/corticall-s2-inputs"import subprocess, sys, os, globfrom pathlib import Pathif not Path("/kaggle/working/tribe-bench").exists():    subprocess.run(["git","clone","--branch",BRANCH,"--depth","50",                    "https://github.com/codesbydevesh/tribe-bench.git",                    "/kaggle/working/tribe-bench"], check=True)os.chdir("/kaggle/working/tribe-bench")print("HEAD:", subprocess.run(["git","log","--oneline","-1"],capture_output=True,text=True).stdout.strip())# find the attached dataset if not givenif DATASET is None:    cands = [p for p in glob.glob("/kaggle/input/*") if Path(p).is_dir()]    for c in cands:        for root in (c, str(Path(c)/"data")):            if (Path(root)/"s2_stimulus.mp4").exists() and (Path(root)/"floc").is_dir():                DATASET = root; break        if DATASET: breakprint("stimulus root:", DATASET or "NOT FOUND — attach the dataset (Input -> Add Input)")assert DATASET, "no attached dataset contains floc/ and s2_stimulus.mp4"os.environ["S2_STIMULUS_ROOT"] = DATASET

In [ ]:
# deps. tribev2 brings neuralset/neuraltrain/exca.!pip install -q -e . 2>&1 | tail -2!pip install -q git+https://github.com/facebookresearch/tribev2.git 2>&1 | tail -3

## 2 · Verify the uploaded inputsCPU-only, read-only. Every file must be byte-identical to what the manifest records.**8/8 must pass.** Anything else means re-upload — do not continue.

In [ ]:
!python3 scripts/s2_verify_inputs.py --stimulus-root $S2_STIMULUS_ROOT

## 3 · Frame-sampling check — the decision pointDoes `neuralset` pick V-JEPA's 64 frames by **timestamp** or by **frame index**?* timestamp → 8 fps is fine, continue* index → 64 frames at 8 fps span **8 s instead of 4**, which halves each image's weight  and smears it into neighbouring events. **STOP.**

In [ ]:
rc = subprocess.run([sys.executable,"scripts/s2_check_frame_sampling.py"]).returncodeprint({0:"TIMESTAMP — safe to continue",       1:"INDEX — STOP. Re-render at 16 fps locally, re-upload.",       2:"AMBIGUOUS — resolve by hand before running."}.get(rc, f"rc={rc}"))FRAME_OK = (rc == 0)

## 4 · Go / no-goRe-derives every checklist item. Expect **GPU GO**.

In [ ]:
assert FRAME_OK, "frame-sampling check did not pass — do not run S2"!python3 scripts/s2_go_no_go.py --review-clean --neuralset-timestamp

## 5 · Run S2One forward pass over the 1050 s stimulus. Both lags are scored from the sametimecourse, so this resolves the t=5 vs t=0 question without a second run.Budget ~3.4 h as an upper bound.

In [ ]:
assert FRAME_OK, "frame-sampling check did not pass — do not run S2"!python3 scripts/s2_run.py --infer --stimulus-root $S2_STIMULUS_ROOT

## 6 · ComplianceChecks the report against the frozen design, including that anyfired stop rule really did fail at *both* lags.

In [ ]:
!python3 scripts/s2_check_compliance.py data/s2_report.json

## 7 · Save the outputsDownload these before the session expires.

In [ ]:
import shutilfor f in ("data/s2_report.json","data/s2_manifest.json"):    if Path(f).exists():        shutil.copy(f, "/kaggle/working/"+Path(f).name); print("saved", Path(f).name)